# Thực hành ở nhà Transformers

Hoàn thiện hàm huấn luyện cho mạng Transformer và tiến hành huấn luyện mô hình

### Cài đặt giải thuật tối ưu và huấn luyện mô hình

In [ ]:
import time
import torch
import torch.nn.functional as F
import numpy as np

# Define the Opt class to store configurations
class Opt:
    def __init__(self):
        self.epochs = 2
        self.d_model = 512
        self.n_layers = 6
        self.heads = 8
        self.dropout = 0.1
        self.batchsize = 1500
        self.printevery = 100
        self.lr = 0.0001
        self.max_strlen = 80
        self.checkpoint = 0
        self.no_cuda = False
        self.load_weights = None
        self.src_data = ""
        self.trg_data = ""
        self.src_lang = ""
        self.trg_lang = ""
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.optimizer = None
        self.train = []

def create_masks(src, trg, opt):
    src_mask = (src != 0).unsqueeze(-2)
    if trg is not None:
        trg_mask = (trg != 0).unsqueeze(-2)
        size = trg.size(1)
        np_mask = np.triu(np.ones((1, size, size)), k=1).astype('uint8')
        np_mask =  torch.from_numpy(np_mask) == 0
        trg_mask = trg_mask & np_mask.to(opt.device)
    else:
        trg_mask = None
    return src_mask, trg_mask

def train_model(model, opt):
    print("Training started...")
    model.train()
    start = time.time()

    for epoch in range(opt.epochs):
        total_loss = 0
        for i, batch in enumerate(opt.train):
            src = batch.src.to(opt.device)
            trg = batch.trg.to(opt.device)
            trg_input = trg[:, :-1]
            targets = trg[:, 1:].contiguous().view(-1)

            # Create masks for transformer
            src_mask, trg_mask = create_masks(src, trg_input, opt)

            preds = model(src, trg_input, src_mask, trg_mask)

            opt.optimizer.zero_grad()
            loss = F.cross_entropy(preds.view(-1, preds.size(-1)), targets, ignore_index=0)
            loss.backward()
            opt.optimizer.step()

            total_loss += loss.item()

            if (i + 1) % opt.printevery == 0:
                avg_loss = total_loss / opt.printevery
                elapsed = time.time() - start
                print("Epoch %d, Iteration %d, Loss: %.4f, Time: %.2fs" % (epoch + 1, i + 1, avg_loss, elapsed))
                total_loss = 0
                start = time.time()

        if opt.checkpoint > 0:
             torch.save(model.state_dict(), f'weights/model_weights_epoch_{epoch+1}.pt')

def main():
    opt = Opt()
    opt.src_data = "data/english.txt"
    opt.trg_data = "data/french.txt"
    if opt.checkpoint > 0:
        print("model weights will be saved every %d minutes")
    print("Configuration and training logic initialized.")

if __name__ == "__main__":
    main()

Configuration and training logic initialized.


In [ ]:
import torch.nn as nn

class DummyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src, trg, src_mask, trg_mask):
        # Simplified forward pass for demonstration
        embedded = self.embedding(src)
        return self.fc_out(embedded)

class DummyBatch:
    def __init__(self, src, trg):
        self.src = src
        self.trg = trg

def create_dummy_data(opt):
    # Generate 5 batches of random token data
    batches = []
    for _ in range(5):
        src = torch.randint(1, 100, (16, 10)) # (batch_size, seq_len)
        trg = torch.randint(1, 100, (16, 11))
        batches.append(DummyBatch(src, trg))
    return batches

In [ ]:
def run_demo():
    opt = Opt()
    opt.epochs = 1
    opt.printevery = 1
    vocab_size = 100

    # Initialize model and dummy data
    model = DummyTransformer(vocab_size, opt.d_model).to(opt.device)
    opt.train = create_dummy_data(opt)
    opt.optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr)

    # Start training
    train_model(model, opt)

if __name__ == "__main__":
    run_demo()

Training started...
Epoch 1, Iteration 1, Loss: 4.7737, Time: 0.01s
Epoch 1, Iteration 2, Loss: 4.7645, Time: 0.00s
Epoch 1, Iteration 3, Loss: 4.7990, Time: 0.00s
Epoch 1, Iteration 4, Loss: 4.7047, Time: 0.00s
Epoch 1, Iteration 5, Loss: 4.7334, Time: 0.00s


### Analysis of Training Implementation

The implementation successfully addresses several critical aspects of training a Transformer model:

1. **Shifted Targets**: By setting `trg_input = trg[:, :-1]` and `targets = trg[:, 1:]`, we implement Teacher Forcing. This ensures the model predicts the next token based on all previous correct tokens.
2. **Masking Logic**:
   - `src_mask` identifies padding in the source to prevent the encoder from attending to it.
   - `trg_mask` (combined with a causal mask, though simplified here) prevents the decoder from 'seeing the future'.
3. **Loss Calculation**: Using `F.cross_entropy` with `ignore_index=0` is essential for handling variable-length sequences, ensuring the model isn't penalized for padding tokens.
4. **Scalability**: The `Opt` class structure allows for easy adjustment of hyperparameters like `d_model`, `heads`, and `batchsize` for real-world datasets like English-French translation.

### Detailed Insights into Transformer Training

Following the implementation of the `train_model` function, here are the key insights and answers to the training requirements:

1. **Data Masking Strategy**:
   - **Source Mask (`src_mask`)**: This is used to ignore padding tokens in the input sequence. It ensures that the self-attention mechanism does not attend to meaningless padding, which could bias the gradients.
   - **Target Mask (`trg_mask`)**: In addition to padding, the target sequence requires a 'subsequent mask' (causal mask). This ensures that during training, the model can only see previous tokens and not 'peek' at future tokens it is supposed to predict.

2. **Target Shifting**:
   - The `trg_input` is defined as `trg[:, :-1]`, while the `targets` (labels) are `trg[:, 1:]`. This shift is crucial because at each time step, the model uses the tokens up to $t$ to predict the token at $t+1$.

3. **Loss Function and Optimization**:
   - We use **Cross-Entropy Loss**. It is important to set `ignore_index=0` (assuming 0 is the padding token ID) so that the loss is not calculated based on padding predictions.
   - The `preds.view(-1, preds.size(-1))` operation flattens the output into a 2D tensor where rows are individual tokens and columns are vocabulary logits, allowing for efficient batch loss calculation.

4. **Training Dynamics**:
   - **Epochs and Iterations**: The model iterates through the entire dataset multiple times (`opt.epochs`). We track the `total_loss` and print an average every `opt.printevery` steps to monitor convergence.